# C3 — Baseline e Regressão Logística

Executa `DummyClassifier` (piso) e `LogisticRegression` na partição
persistida pelo Notebook 03, usando **somente treino e validação**.

Sem tuning, sem calibração, sem escolha de limiar. O teste permanece
fechado. Limiar 0.5 é apenas diagnóstico.

**Pré-requisito:** `pip install -e .`

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import load_supervised_artifact, split_partitions
from src.evaluation.leakage import check_leakage
from src.evaluation.metrics import metricas_completas
from src.models.baseline import evaluate_baseline
from src.models.logistic import build_logistic_pipeline

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR      = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print(C.resumo())

## Carregar artefacto e materializar partições

`load_supervised_artifact` valida schema, alvo binário e presença das
três partições. `split_partitions` devolve apenas `(treino, validacao)`
— o teste fica inacessível por construção.

In [ ]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "provisional").lower()

df, manifest = load_supervised_artifact(ROOT_DIR, expected_kind=COHORT_KIND)
treino, validacao = split_partitions(df, manifest)

PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

X_train = treino[FEATURES].copy()
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")
X_val   = validacao[FEATURES].copy()
y_val   = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

check_leakage(X_train)
check_leakage(X_val)

n_teste = int((df[PARTICAO].astype(str) == "teste").sum())

display(pd.DataFrame({
    "particao":   ["treino", "validacao"],
    "n":          [len(treino), len(validacao)],
    "positivos":  [int(y_train.sum()), int(y_val.sum())],
    "prevalencia":[float(y_train.mean()), float(y_val.mean())],
}))

print(f"cohort_kind : {manifest['cohort_kind']}")
print(f"features    : {len(FEATURES)}")
print(f"teste       : {n_teste:,} (fechado)")

## Baseline e Regressão Logística

`DummyClassifier` com a estratégia do YAML (`prior` por omissão) define
o piso. A logística usa `C=1.0` como ponto de partida — sem tuning.

In [ ]:
log_cfg = C.cfg("models", "logistic") or {}

metricas_dummy = evaluate_baseline(
    X_train, y_train, X_val, y_val,
    strategy=C.cfg("models", "baseline", "strategy") or "prior",
)

pipe_log = build_logistic_pipeline(
    C=1.0,
    solver=log_cfg.get("solver", "lbfgs"),
    max_iter=int(log_cfg.get("max_iter", 1000)),
    class_weight=log_cfg.get("class_weight", "balanced"),
)
pipe_log.fit(X_train, y_train)
prob_val = pipe_log.predict_proba(X_val)[:, 1]

metricas_log = {**metricas_completas(y_val, prob_val, limiar=0.5), "modelo": "logistica_C1"}

display(pd.DataFrame([metricas_dummy, metricas_log]))

## Comparação e persistência

Gravamos comparação, predições de validação e manifesto em `outputs/`.
Nada é finalista — a decisão pertence a C4/C5.

In [ ]:
CAMPOS = ("auc", "ap", "brier", "f1", "precision", "recall", "accuracy")

comparacao = (
    pd.DataFrame([
        {"modelo": "dummy_prior",   **{k: metricas_dummy[k] for k in CAMPOS}},
        {"modelo": "logistica_C1", **{k: metricas_log[k]   for k in CAMPOS}},
    ])
    .sort_values("ap", ascending=False)
    .reset_index(drop=True)
)

delta_ap = metricas_log["ap"] - metricas_dummy["ap"]
display(comparacao)
print(f"Δ AP (logística − dummy): {delta_ap:+.4f}")

pred_val = pd.DataFrame({
    "y_true":      y_val.to_numpy(),
    "y_prob":      prob_val,
    "modelo":      "logistica_C1",
    "cohort_kind": manifest["cohort_kind"],
})
pred_val["y_pred"] = (pred_val["y_prob"] >= 0.5).astype(int)

In [ ]:
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo   = f"C3_baseline_logistica_{manifest['cohort_kind']}_{timestamp}"

caminho_comparacao = TABLES_DIR      / f"{prefixo}_comparacao.csv"
caminho_predicoes  = PREDICTIONS_DIR / f"{prefixo}_predicoes_validacao.parquet"
caminho_manifesto  = TABLES_DIR      / f"{prefixo}_manifesto.json"

comparacao.to_csv(caminho_comparacao, index=False, encoding="utf-8-sig")
pred_val.to_parquet(caminho_predicoes, index=False)

manifesto_c3 = {
    "cohort_kind":        manifest["cohort_kind"],
    "dataset_path":       manifest["dataset_path"],
    "dataset_sha256":     manifest["dataset_sha256"],
    "feature_columns":    FEATURES,
    "target_column":      ALVO,
    "partition_column":   PARTICAO,
    "n_treino":           int(len(treino)),
    "n_validacao":        int(len(validacao)),
    "n_teste_fechado":    n_teste,
    "limiar_usado":       0.5,
    "limiar_e_diagnostico": True,
    "tuning_realizado":   False,
    "calibracao_aplicada":False,
    "teste_aberto":       False,
    "modelos":            ["dummy_prior", "logistica_C1"],
    "artefatos": {
        "comparacao":          str(caminho_comparacao),
        "predicoes_validacao": str(caminho_predicoes),
    },
}
caminho_manifesto.write_text(
    json.dumps(manifesto_c3, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Comparação : {caminho_comparacao}")
print(f"Predições  : {caminho_predicoes}")
print(f"Manifesto  : {caminho_manifesto}")

In [ ]:
with Run(
    modelo="C3_baseline_logistica",
    parametros={
        "cohort_kind":          manifest["cohort_kind"],
        "logistic_C":           1.0,
        "logistic_solver":      log_cfg.get("solver", "lbfgs"),
        "limiar_diagnostico":   0.5,
        "tuning_realizado":     False,
        "calibracao_aplicada":  False,
        "teste_aberto":         False,
    },
    dataset={
        "n_treino":        int(len(treino)),
        "n_validacao":     int(len(validacao)),
        "n_teste_fechado": n_teste,
    },
    features={"n_features": len(FEATURES), "usadas": FEATURES},
    notas="C3 — baseline e logística na partição persistida. Teste não consultado.",
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(
        dummy_ap=metricas_dummy["ap"],
        dummy_auc=metricas_dummy["auc"],
        logistica_ap=metricas_log["ap"],
        logistica_auc=metricas_log["auc"],
        logistica_brier=metricas_log["brier"],
    )

print("Run registrado em:", r.pasta)